# CrewAI: a team of roles

CrewAI describes a multi-agent system as a team. Agents have roles, tasks describe work, and a process decides how tasks move.

In this notebook, we will:

- see why this chapter uses a separate environment;
- connect the same course model;
- define two roles and their tools;
- pass work through a sequential crew; and
- decide when the team metaphor helps.

CrewAI needs its own environment. Set it up once with these commands:

    python -m venv .venv-crewai && . .venv-crewai/bin/activate
    pip install -e ".[crewai]"
    python -m ipykernel install --user --name agentic-lab-crewai

Then select the agentic-lab-crewai kernel for this notebook.

In [ ]:
import importlib.metadata as metadata
import os
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
TEMPERATURE = 0
litellm.drop_params = True

os.environ["CREWAI_TELEMETRY_OPT_OUT"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

## 1. Dependency pins can force separate environments

Python packages often require specific versions of shared libraries. CrewAI's requirements do not fit every version in the main course environment.

A separate environment lets both sets of requirements stay valid.

In [ ]:
from packaging.requirements import Requirement

important = {"mcp", "pydantic", "openai"}
requirements = [
    Requirement(text)
    for text in metadata.requires("crewai") or []
    if Requirement(text).name in important
]

for requirement in requirements:
    installed = metadata.version(requirement.name)
    print(requirement.name, "required:", requirement.specifier, "installed:", installed)

## 2. Connect the model

CrewAI's LLM uses a LiteLLM model string, so the same openrouter-prefixed name works here.

In [ ]:
from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool

crew_model = LLM(model=MODEL, temperature=TEMPERATURE)

print("provider:", crew_model.provider)
print("model:", crew_model.model)

## 3. Give each role its own tools

The triager reads order and customer data. The policy checker searches policy. Their permissions match their jobs.

In [ ]:
from shoplab import world

orders = {item["order_id"]: item for item in world.load_orders()}
customers = {item["customer_id"]: item for item in world.load_customers()}

@tool("get_order")
def get_order(order_id: str) -> dict:
    """Look up one Larkspur order."""
    return orders.get(order_id, {"error": "order not found"})

@tool("get_customer")
def get_customer(customer_id: str) -> dict:
    """Look up one Larkspur customer."""
    return customers.get(customer_id, {"error": "customer not found"})

@tool("find_policy")
def find_policy(query: str) -> list:
    """Search Larkspur policy documents."""
    return world.search_policy(query)

print(get_order.name, get_customer.name, find_policy.name)

## 4. Define two roles

A CrewAI Agent is a model with a role, goal, story, and tools.

In [ ]:
triager = Agent(
    role="Returns triager",
    goal="Gather the order and customer facts and propose a return decision.",
    backstory="You work at the Larkspur operations desk.",
    tools=[get_order, get_customer],
    llm=crew_model,
    allow_delegation=False,
    max_iter=6,
    verbose=False,
)

checker = Agent(
    role="Policy checker",
    goal="Check the proposed decision against the exact shop policy.",
    backstory="You protect the desk from unsupported decisions.",
    tools=[find_policy],
    llm=crew_model,
    allow_delegation=False,
    max_iter=6,
    verbose=False,
)

print(triager.role, "->", checker.role)

## 5. Connect tasks in sequence

The first task gathers facts. The second receives that result as context and checks it.

In [ ]:
TICKET = (
    "TKT-2205: order ORD-7312, customer CUST-07. Opened Torrent boots were "
    "returned after 18 days for a refund."
)

triage_task = Task(
    description="Use get_order and get_customer. Propose a decision and amount for " + TICKET,
    expected_output="A short proposal with item value, customer tier, decision, and amount.",
    agent=triager,
)

check_task = Task(
    description="Check the proposal with find_policy. Correct it if needed.",
    expected_output="One line: decision | policy_id | amount.",
    agent=checker,
    context=[triage_task],
)

crew = Crew(
    agents=[triager, checker],
    tasks=[triage_task, check_task],
    process=Process.sequential,
    verbose=False,
)

print("process:", crew.process)

In [ ]:
result = await crew.kickoff_async()

for output in result.tasks_output:
    print("role:", output.agent)
    print(output.raw, "\n")

print("known: partial_refund | pol-restocking | 170.99")
print("tokens:", result.token_usage.total_tokens)

## 6. Sequential and hierarchical processes

A sequential process follows the task order we set. A hierarchical process gives a manager model the job of assigning work.

In [ ]:
manager_crew = Crew(
    agents=[triager, checker],
    tasks=[Task(
        description="Triage TKT-2205 by delegating fact gathering and policy checking.",
        expected_output="One line: decision | policy_id | amount.",
    )],
    process=Process.hierarchical,
    manager_llm=crew_model,
    verbose=False,
)

print("process:", manager_crew.process)
print("manager model set:", manager_crew.manager_llm is not None)

Use a crew when roles truly need separate tools, goals, or ownership. If one agent can do the work clearly, extra roles add model calls and another handoff where details can be lost.

## What to remember

1. Dependency conflicts may require a separate environment.
2. A CrewAI Agent combines role, goal, model, and tools.
3. A Task describes work and expected output.
4. Sequential process follows a fixed task order.
5. Hierarchical process lets a manager assign work.
6. A team is useful only when the role boundary is real.

## Small practice steps

1. Print the installed CrewAI version.
2. Give the checker a stricter expected output.
3. Remove one role and describe what becomes simpler.